# CLaRa Apple-Native Evaluation

This notebook evaluates Apple's pretrained CLaRa-7B-E2E checkpoint using **Apple's own `modeling_clara.py`**,
patched for 4-bit NF4 quantization to fit on T4 GPUs.

### Why a new evaluation pipeline?

The repo's reimplementation (`models/clara_model.py`) had **5 critical architectural mismatches** with Apple's original code:

| Bug | Impact |
|-----|--------|
| Wrong compression: embedding concat vs memory token injection | Memory tokens contain zero document info |
| Wrong adapter names (`compressor/query/generator` vs `encoder/query_reasoner/decoder`) | Incorrect weight loading |
| Wrong LoRA targets (attention-only vs `all-linear`) | ~50% of trained weights silently dropped |
| Wrong prompt format (no system prompt, no memory token placeholders) | Model never saw this format during training |
| Missing special token embeddings from `decoder_first_last_layers.pth` | Random/zero embeddings for MEM tokens |

This notebook bypasses all of that by using Apple's original code directly via `trust_remote_code=True`.

### Prerequisites

- Kaggle dataset: `tokiggle/clara-7b-e2e` attached as input
- GPU: T4 (Kaggle free tier)
- `bitsandbytes` installed

---
## Section 1 — Setup

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 │ Install dependencies & verify environment
# ═══════════════════════════════════════════════════════════════════════════════

!pip install bitsandbytes --no-cache-dir -q

import os, torch, gc

APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e/compression-16"

assert os.path.isdir(APPLE_CKPT), f"Checkpoint not found: {APPLE_CKPT}"
assert torch.cuda.is_available(), "GPU not available"

print(f"✓ Checkpoint: {APPLE_CKPT}")
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"✓ Files: {os.listdir(APPLE_CKPT)}")

---
## Section 2 — Quick Test (5 samples)

Run a fast evaluation on 5 SQuAD samples to verify the model is generating sensible answers.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 │ Quick Test: Apple Native Eval (5 SQuAD samples)
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e/compression-16"

print("╔" + "═" * 60 + "╗")
print("║  QUICK TEST — Apple Native Pipeline (5 SQuAD samples)       ║")
print("╚" + "═" * 60 + "╝")

eval_env = os.environ.copy()
eval_env.update({
    "CLARA_CKPT_PATH"      : APPLE_CKPT,
    "CLARA_DATASET"        : "squad",
    "CLARA_EVAL_MODE"      : "oracle",
    "CLARA_EVAL_BS"        : "1",
    "CLARA_N_VAL"          : "5",
    "CLARA_MAX_NEW_TOKENS" : "32",
    "CLARA_MODEL_VERSION"  : "QuickTest_AppleNative",
})

subprocess.run(
    ["python", "-m", "scripts.evaluate_apple"],
    env=eval_env, check=True,
)

print("\n✅ Quick test complete! Check the predictions above.")
print("If they look sensible, run the full evaluation in Section 3.")

### Manual Inference Test

Load the model directly and run a few hand-crafted examples for qualitative inspection.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 │ Manual Inference: Load model + run custom Q&A examples
# ═══════════════════════════════════════════════════════════════════════════════

import torch, gc, os, sys
from transformers import AutoModel

# Add repo root to path
REPO_ROOT = os.path.dirname(os.path.abspath("__file__"))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Assemble workdir using our helper
from scripts.evaluate_apple import assemble_workdir

APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e/compression-16"
work_dir = assemble_workdir(APPLE_CKPT)

gc.collect(); torch.cuda.empty_cache()
model = AutoModel.from_pretrained(
    work_dir, trust_remote_code=True, load_pretrained_checkpoint=True,
)
model.to("cuda")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# ── Test cases ────────────────────────────────────────────────────────────────
TEST_CASES = [
    {
        'docs': [
            'The Battle of Hastings was fought on 14 October 1066 between '
            'the Norman-French army of William, the Duke of Normandy, and '
            'an English army under the Anglo-Saxon King Harold Godwinson.',
        ],
        'q': 'When was the Battle of Hastings fought?',
        'expected': '14 October 1066',
    },
    {
        'docs': [
            'Weldenia is a monotypic genus of flowering plants in the family '
            'Commelinaceae, native to Mexico and Guatemala.',
        ],
        'q': 'Which genus grows originally in Mexico and Guatemala, Phylica or Weldenia?',
        'expected': 'Weldenia',
    },
    {
        'docs': [
            'Albert Einstein was born on 14 March 1879 in Ulm, in the '
            'Kingdom of Württemberg in the German Empire. He developed the '
            'theory of relativity.',
        ],
        'q': 'Where was Albert Einstein born?',
        'expected': 'Ulm',
    },
    {
        'docs': [
            'Python is a high-level, general-purpose programming language. '
            'Its design philosophy emphasizes code readability. '
            'Python was conceived in the late 1980s by Guido van Rossum.',
        ],
        'q': 'Who created Python?',
        'expected': 'Guido van Rossum',
    },
]

print("═" * 65)
print("  MANUAL INFERENCE TEST  (Apple E2E pipeline)")
print("═" * 65)

with torch.no_grad():
    for i, tc in enumerate(TEST_CASES, 1):
        decoded, _ = model.generate_from_questions(
            questions=[tc['q']],
            documents=[tc['docs']],
            max_new_tokens=64,
        )
        pred = decoded[0]
        match = tc['expected'].lower() in pred.lower()

        print(f"\n[{i}] Question : {tc['q']}")
        print(f"    Expected : {tc['expected']}")
        print(f"    Model    : {pred[:200]}")
        print(f"    Match    : {'✓ YES' if match else '✗ NO'}")

print("\n" + "═" * 65)

# Free memory before full eval
del model; gc.collect(); torch.cuda.empty_cache()
print("Model unloaded. Ready for full evaluation.")

---
## Section 3 — Full Evaluation (TriviaQA + SQuAD)

Evaluates the Apple CLaRa-7B-E2E checkpoint on 500 validation samples from each dataset.
Results are saved to `results/eval_scores.csv`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 │ Full Apple Native Eval — TriviaQA + SQuAD (500 samples each)
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e/compression-16"

for dataset in ["triviaqa", "squad"]:
    print("\n" + "╔" + "═" * 60 + "╗")
    print(f"║  MODEL A — Apple Native Eval: {dataset.upper():<30}║")
    print("╠" + "═" * 60 + "╣")
    print("║  Pipeline  : Apple modeling_clara.py (4-bit NF4)            ║")
    print(f"║  Dataset   : {dataset:<47}║")
    print("║  Eval mode : oracle  |  Metrics: EM + F1                    ║")
    print("╚" + "═" * 60 + "╝")

    eval_env = os.environ.copy()
    eval_env.update({
        "CLARA_CKPT_PATH"      : APPLE_CKPT,
        "CLARA_DATASET"        : dataset,
        "CLARA_EVAL_MODE"      : "oracle",
        "CLARA_EVAL_BS"        : "1",
        "CLARA_N_VAL"          : "500",
        "CLARA_MAX_NEW_TOKENS" : "32",
        "CLARA_MODEL_VERSION"  : f"ModelA_AppleNative_{dataset}",
    })

    subprocess.run(
        ["python", "-m", "scripts.evaluate_apple"],
        env=eval_env, check=True,
    )

print("\n✅ Apple native evaluation complete (both datasets).")
print("Results appended to: results/eval_scores.csv")

---
## Section 4 — Results Summary

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 │ Results Summary
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd

CSV_PATH = "results/eval_scores.csv"

if not os.path.exists(CSV_PATH):
    print(f"No results found at {CSV_PATH}. Run evaluation cells first.")
else:
    df = pd.read_csv(CSV_PATH)

    print("═" * 80)
    print("  CLaRa EXPERIMENT RESULTS SUMMARY")
    print("═" * 80)
    print()

    display_df = df[[
        'Model_Version', 'Dataset', 'Eval_Mode',
        'Exact_Match(%)', 'F1_Score(%)', 'Timestamp'
    ]].copy()
    display_df = display_df.sort_values(['Dataset', 'Model_Version'])

    pd.set_option('display.max_colwidth', 45)
    pd.set_option('display.width', 120)
    print(display_df.to_string(index=False))

    print()
    print("─" * 80)
    print("PAPER REFERENCE (Table 2 — Oracle, CLaRa-Mistral-7B 16×):")
    print("  NQ        : EM=63.29%  F1=71.54%")
    print("  HotpotQA  : EM=57.54%  F1=71.17%")
    print("  (Instruction-tuned init, Normal setting)")
    print("─" * 80)

---
## Experimental Notes

### Why Apple Native Pipeline?

The repo's `models/clara_model.py` had critical architectural mismatches:

1. **Compression**: Used embedding concatenation instead of memory token injection
2. **Adapter names**: `compressor/query/generator` vs Apple's `encoder_adapter/query_reasoner_adapter/decoder_adapter`
3. **LoRA targets**: Attention-only vs `all-linear` (drops ~50% of trained weights)
4. **Prompt format**: Missing system prompt and memory token placeholders
5. **Special tokens**: `decoder_first_last_layers.pth` embeddings not loaded

Using Apple's original code with `trust_remote_code=True` bypasses all of these.

### T4 GPU Adaptations

| Setting | Paper | This notebook |
|---------|-------|---------------|
| Quantization | BF16 (8×H100) | NF4 4-bit (T4) |
| `generation_top_k` | 5 | 5 (same) |
| `doc_max_length` | 256 | 256 (same) |

### Citation

```bibtex
@article{he2026clara,
  title   = {CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning},
  author  = {He, Jie and Bai, Richard He and Williamson, Sinead and Pan, Jeff Z. 
             and Jaitly, Navdeep and Zhang, Yizhe},
  journal = {arXiv preprint arXiv:2511.18659},
  year    = {2026}
}
```